In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import os 

df = pd.read_csv("data/reworked_mock_data.csv")
df.columns = df.columns.str.strip().str.lower()
def clean_dataset(df):
    
    df['sleep_hours'] = pd.to_timedelta(df['sleep_duration']).dt.total_seconds() / 3600.0
    
    
    df['target'] = (df['result'] == 'Win').astype(int)
    return df


env_cols = ['avg_lumen', 'avg_celsius', 'avg_ppm', 'avg_ml']
game_cols = [
    'rating_diff', 'total_ply', 'opening_ply', 'player_move_count', 
    'duration_min', 'player_opening_win_rate', 'sleep_hours'
]
cat_cols = ['eco_code', 'time_control', 'is_berserk', 'is_player_piece_black']


def apply_dual_clustering(X_train, X_test):
    scaler_env = StandardScaler()
    env_train_scaled = scaler_env.fit_transform(X_train[env_cols])
    km_env = KMeans(n_clusters=3, random_state=42).fit(env_train_scaled)
    
    scaler_game = StandardScaler()
    game_train_scaled = scaler_game.fit_transform(X_train[game_cols])
    km_game = KMeans(n_clusters=4, random_state=42).fit(game_train_scaled)
    
   
    for df, s_env, s_game in [(X_train, scaler_env, scaler_game), (X_test, scaler_env, scaler_game)]:
        df['env_cluster'] = km_env.predict(s_env.transform(df[env_cols]))
        df['game_cluster'] = km_game.predict(s_game.transform(df[game_cols]))
    
    return X_train, X_test


preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols + ['env_cluster', 'game_cluster'])
    ],
    remainder='passthrough'
)

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
])


df = pd.read_sql("SELECT * FROM games", conn)
df = clean_dataset(df)
X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1), df['target'])
X_train, X_test = apply_dual_clustering(X_train, X_test)
model_pipeline.fit(X_train, y_train)